In [2]:
import sys
print('Kernel Python:', sys.executable)

Kernel Python: /workspaces/llm-playground/02-vector-search/homework/.venv/bin/python


In [3]:
from embedder import Embedder

2026-06-29 21:05:03.260276357 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Q1. Embedding a query

In [4]:
# Same pattern as the pgvector notebook: build model -> embed query
embedder = Embedder()
query = 'How does approximate nearest neighbor search work?'
query_vector = embedder.encode(query)

In [5]:
query_vector.shape

(384,)

In [6]:
query_vector[0]

np.float64(-0.02058203437252893)

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [8]:
len(documents)

72

In [9]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

Q2. Cosine similarity


In [10]:
# pick the target lesson page
target_filename = "07-sqlitesearch-vector.md"
page = next(doc for doc in documents if doc["filename"].endswith(target_filename))

# embed document text and compute cosine similarity via dot product
content = page["content"]
content_vector = embedder.encode(content)
score = query_vector.dot(content_vector)

score

np.float64(0.36107026789538205)

Q3. Chunking and search by hand


1. Chunk lessons into overlapping pieces.
2. Embed chunk content into matrix X.
3. Score chunks with Q1 query vector and take the top match.

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [12]:
import numpy as np

chunk_texts = [chunk["content"] for chunk in chunks]
chunk_vectors = embedder.encode_batch(chunk_texts)
X = np.array(chunk_vectors)
X.shape

(295, 384)

In [13]:
v = query_vector
scores = X.dot(v)

best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]

best_chunk["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

Q4. Vector search with minsearch


1. Build a minsearch vector index from chunk embeddings.
2. Encode the Q4 query and run vector search.
3. Return the first result filename.

In [14]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [15]:
query_q4 = "What metric do we use to evaluate a search engine?"
v_q4 = embedder.encode(query_q4)

results_q4 = vindex.search(v_q4, num_results=5)
results_q4[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

Q5. Text search vs vector search


1. Build a text index on chunk content.
2. Build a vector index on chunk embeddings.
3. Compare top-5 filenames for text and vector search.

In [27]:
from minsearch import Index, VectorSearch

# Keyword index: ranks chunks by text overlap in content.
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

# Vector index: uses precomputed chunk embeddings X for semantic search.
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

In [17]:
query_q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embedder.encode(query_q5)



In [29]:
# Top-5 keyword matches for the raw query text.
top5_text = text_index.search(query_q5, num_results=5)
top5_text_filenames = [item["filename"] for item in top5_text]

top5_text_filenames

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [ ]:
# Score every chunk vector against the query vector (cosine-like similarity).
scores_q5 = X.dot(v_q5)
# Same ordering method as vector_search.ipynb: argsort -> take last 5 -> reverse.
top5_idx = np.argsort(scores_q5)[-5:]
top5_idx = top5_idx[::-1]

top5_vector = [chunks[i] for i in top5_idx]
top5_vector_scores = scores_q5[top5_idx]

In [25]:
top5_vector_filenames = [item["filename"] for item in top5_vector]

list(zip(top5_vector_scores, top5_vector_filenames))

[(np.float64(0.5524407784220435), '02-vector-search/lessons/08-pgvector.md'),
 (np.float64(0.5305404520455319), '02-vector-search/lessons/08-pgvector.md'),
 (np.float64(0.4685658347141549), '03-orchestration/lessons/05-rag.md'),
 (np.float64(0.4643922431223802), '02-vector-search/lessons/08-pgvector.md'),
 (np.float64(0.4490358390495504), '02-vector-search/lessons/08-pgvector.md')]

In [28]:
# Compare top-5 filenames from both methods.
comparison = {"text": top5_text_filenames, "vector": top5_vector_filenames}

# Files retrieved by vector search but missing from keyword top-5.
vector_only = sorted(set(top5_vector_filenames) - set(top5_text_filenames))

comparison, vector_only

({'text': ['02-vector-search/lessons/02-embeddings.md',
   '03-orchestration/lessons/05-rag.md',
   '02-vector-search/lessons/01-intro.md',
   '03-orchestration/lessons/05-rag.md',
   '02-vector-search/lessons/01-intro.md'],
  'vector': ['02-vector-search/lessons/08-pgvector.md',
   '02-vector-search/lessons/08-pgvector.md',
   '03-orchestration/lessons/05-rag.md',
   '02-vector-search/lessons/08-pgvector.md',
   '02-vector-search/lessons/08-pgvector.md']},
 ['02-vector-search/lessons/08-pgvector.md'])

Q6. Hybrid search


In [30]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [31]:
query_q6 = "How do I give the model access to tools?"
v_q6 = embedder.encode(query_q6)

# Get ranked candidates from both retrieval methods.
vector_results = vector_index.search(v_q6, num_results=5)
text_results = text_index.search(query_q6, num_results=5)

# Fuse both ranked lists with reciprocal rank fusion.
results = rrf([vector_results, text_results])

results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'